In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

from sklearn.ensemble import RandomForestClassifier as RFC

In [ ]:
rkf = RepeatedStratifiedKFold(n_splits=8, n_repeats=10, random_state=42)

# PCA
N_COMPONENTS_OPTIONS = [5]

# ESTIMATOR
N_ESTIMATOR_OPTIONS = [50, 75, 100, 125, 150, 175, 200]
MIN_SAMPLES_LEAF_OPTIONS = [1, 2]

# 1. Definizione Pipeline #
pipe = Pipeline([
    # Step 1: Scaling 
    ("scaling", RobustScaler()),  
    
    # Step 2: Classificatore BOOTSTRAP SEMPRE TRUE PERCHÉ HO POCHI SAMPLES (44)
    ("classify", RFC(
        random_state=42, 
        bootstrap=True, 
        criterion='gini',
        max_features = 'sqrt',
        max_depth=4,
        )
    ) 
])

# 2. Definizione griglia dei parametri #
param_grid = {
    # Per provare parametri del classificatore
    "classify__n_estimators": N_ESTIMATOR_OPTIONS,
    "classify__min_samples_leaf": MIN_SAMPLES_LEAF_OPTIONS, # Stabilizza le foglie
}

# 3. Configurazione GridSearch #
grid = GridSearchCV(
    pipe, 
    param_grid=param_grid, 
    cv=rkf, # rkf = RepeatedStratifiedKFold(n_splits=8, n_repeats=10, random_state=42)
    n_jobs=-1, # «Number of jobs to run in parallel. -1 means using all processors»
    scoring={
        'score': 'accuracy',
        'sensitivity': 'recall'  # recall = sensitivity
    },
    refit='score', # «For multiple metric evaluation, needs to be a str denoting the
    # scorer to use to find the best parameters for refitting the estimator at the end»
    return_train_score=False
)

# 4. Training e Validation (su Segnale B) #
grid.fit(signal_b, labels)

# 5. Risultati #
print(f"La miglior configurazione: {grid.best_params_}")
print(f"Fornisce accuracy in validation: {grid.best_score_:.4f}")

# 6. Test su segnale A #
accuracy_finale = grid.score(signal_a, labels)
print(f"Risultato sul set indipendente (Segnale A): {accuracy_finale:.4f}")